Позаимствуем часть кода из версии без кластеризации:

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from find_similar import TimeSeriesSubsequenceSearcher

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)

1) Загрузка данных из CSV файлов (надо прикрутить бенчмарк)

In [2]:
def parse_timestamp_column(values: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(values, errors="coerce")
    unit = "ms" if numeric.abs().max() >= 1e11 else "s"
    return pd.to_datetime(values, unit=unit, utc=True).dt.tz_convert(None)


def trim_last_weeks(df: pd.DataFrame, weeks: int = 6) -> pd.DataFrame:
    timestamps = parse_timestamp_column(df["timestamp"])
    cutoff = timestamps.max() - pd.Timedelta(weeks=weeks)
    return df.loc[timestamps >= cutoff].copy()


paths = sorted(Path(".").glob("time_series*.csv"))
series_dict = {path.stem: trim_last_weeks(pd.read_csv(path), weeks=6) for path in paths}

summary = pd.DataFrame(
    {
        "series_id": list(series_dict),
        "rows": [len(df) for df in series_dict.values()],
        "columns": [", ".join(df.columns) for df in series_dict.values()],
    }
)
summary

,series_id,rows,columns
0,time_series,9216,"timestamp, value_0"
1,time_series_2,1723,"timestamp, value, ground_truth, predicted"
2,time_series_3,1723,"timestamp, value, ground_truth, predicted"
3,time_series_4,1924,"timestamp, value, ground_truth, predicted"
4,time_series_5,1840,"timestamp, value, ground_truth, predicted"
5,time_series_6,1849,"timestamp, value, ground_truth, predicted"


2. Приводим все временные ряды к одинаковой частоте дискретизации 

In [3]:
searcher = TimeSeriesSubsequenceSearcher(
    freq="30min",
    agg="mean",
    interpolate_limit=3,
    normalize=True,
    exclusion_fraction=0.75,
)

searcher.fit(series_dict)

prepared_summary = pd.DataFrame(
    {
        "series_id": list(searcher.prepared_series_),
        "prepared_points": [len(series) for series in searcher.prepared_series_.values()],
        "missing": [int(series.isna().sum()) for series in searcher.prepared_series_.values()],
        "start": [series.index.min() for series in searcher.prepared_series_.values()],
        "end": [series.index.max() for series in searcher.prepared_series_.values()],
    }
)
prepared_summary

,series_id,prepared_points,missing,start,end
0,time_series,1536,0,2025-08-03 00:00:00,2025-09-03 23:30:00
1,time_series_2,2017,71,2025-07-22 17:00:00,2025-09-02 17:00:00
2,time_series_3,2017,71,2025-07-22 17:00:00,2025-09-02 17:00:00
3,time_series_4,2017,32,2025-07-22 21:30:00,2025-09-02 21:30:00
4,time_series_5,2017,60,2025-07-22 15:30:00,2025-09-02 15:30:00
5,time_series_6,2017,41,2025-07-22 13:00:00,2025-09-02 13:00:00


3. Выбыор отрекзка, похожие на который мы будем искать (надо прикрутить нормальный выбор отрезка)

In [4]:
source_series_id = "time_series_2" if "time_series_2" in searcher.prepared_series_ else next(iter(searcher.prepared_series_))
query_points = 24  # 24 points * 30min = 12 hours

source_series = searcher.prepared_series_[source_series_id]
rolling_std = source_series.rolling(query_points).std().to_numpy()

if np.isfinite(rolling_std).any():
    query_end_idx = int(np.nanargmax(rolling_std))
    query_start_idx = query_end_idx - query_points + 1
else:
    query_start_idx = min(max(int(len(source_series) * 0.65), 0), len(source_series) - query_points)
    query_end_idx = query_start_idx + query_points - 1

query_start_idx = max(query_start_idx, 0)
query_end_idx = min(query_start_idx + query_points - 1, len(source_series) - 1)
query_start_time = source_series.index[query_start_idx]
query_end_time = source_series.index[query_end_idx]
query = source_series.iloc[query_start_idx : query_end_idx + 1]

query_start_time, query_end_time, len(query), float(query.std())

(Timestamp('2025-07-26 01:30:00'),
 Timestamp('2025-07-26 13:00:00'),
 24,
 132.7321319070093)

Здесь можно настроить параметры поиска: 
1. Из какого временного ряда исходный отрезок
2. Левый уонец отрезка
3. Правй конец отрезка
4. Сколько похожих мы хотим
5. Учитывать ли исходный временной ряд

In [5]:
results = searcher.search_by_time_range(
    source_series_id=source_series_id,
    start_time=query_start_time,
    end_time=query_end_time,
    top_k=50,
    search_in_source=False,
)

results

,series_id,start_idx,end_idx,start_time,end_time,distance
0,time_series_3,161,184,2025-07-26 01:30:00,2025-07-26 13:00:00,0.000000
1,time_series_5,786,809,2025-08-08 00:30:00,2025-08-08 12:00:00,0.517902
2,time_series_3,1600,1623,2025-08-25 01:00:00,2025-08-25 12:30:00,0.635830
3,time_series_3,881,904,2025-08-10 01:30:00,2025-08-10 13:00:00,0.657821
4,time_series_5,834,857,2025-08-09 00:30:00,2025-08-09 12:00:00,0.686994
5,time_series_5,546,569,2025-08-03 00:30:00,2025-08-03 12:00:00,0.694709
6,time_series_5,1794,1817,2025-08-29 00:30:00,2025-08-29 12:00:00,0.701763
7,time_series_3,929,952,2025-08-11 01:30:00,2025-08-11 13:00:00,0.730510
8,time_series_3,1408,1431,2025-08-21 01:00:00,2025-08-21 12:30:00,0.733567
9,time_series_3,448,471,2025-08-01 01:00:00,2025-08-01 12:30:00,0.753770


In [6]:
import plotly.graph_objects as go
from typing import Iterable
import pandas as pd
import numpy as np


def add_line(
    fig: go.Figure,
    x_values: Iterable,
    y_values: Iterable,
    name: str,
    color: str,
) -> go.Figure:
    trace = go.Scatter(
        x=x_values,
        y=y_values,
        mode="lines",
        name=name,
        line={"color": color},
    )
    fig.add_trace(trace)
    return fig


def update_layout(fig: go.Figure):
    fig.update_layout(
        title="Time Series",
        xaxis_title="Time",
        yaxis_title="Value",
        height=600,
        hovermode="x unified",
        showlegend=True,
        xaxis=dict(
            rangeselector=dict(
                buttons=[
                    dict(count=1, label="day", step="day", stepmode="backward"),
                    dict(count=7, label="week", step="day", stepmode="backward"),
                    dict(count=1, label="month", step="month", stepmode="backward"),
                    dict(step="all"),
                ]
            ),
            rangeslider={"visible": True},
            type="date",
        ),
        yaxis={"fixedrange": False},
    )
    return fig


def plot_time_series(time_series: pd.DataFrame, title: str = "Time Series"):
    fig = go.Figure()
    add_line(fig, time_series.index, time_series["value_0"], title, "blue")
    update_layout(fig)
    return fig


def add_confidence_interval(fig: go.Figure, forecast: pd.DataFrame):
    ds = forecast.index
    fig.add_trace(
        go.Scatter(
            x=ds,
            y=forecast["expected"],
            mode="lines",
            name="Forecast",
            line=dict(color="rgba(31, 119, 180, 0.8)"),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=ds,
            y=forecast["upper"],
            mode="lines",
            name="Upper Bound",
            line=dict(width=0),
            showlegend=False,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=ds,
            y=forecast["lower"],
            mode="lines",
            name="Lower Bound",
            fill="tonexty",
            fillcolor="rgba(31, 119, 180, 0.2)",
            line=dict(width=0),
            showlegend=False,
        )
    )
    return fig


def add_points(
    fig: go.Figure,
    x_values: Iterable,
    y_values: Iterable,
    name: str = "Anomalies",
    color: str = "red",
):
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=y_values,
            mode="markers",
            name=name,
            marker=dict(color=color, size=10),
        )
    )
    return fig


def add_anomalies(
    fig: go.Figure,
    time_series: pd.DataFrame,
    is_anomaly: np.ndarray,
    expected_values: np.array,
    expected_bounds: np.array,
):
    time_series = time_series.copy()
    time_series["expected"] = expected_values
    time_series["upper"] = expected_bounds[:, 0]
    time_series["lower"] = expected_bounds[:, 1]
    anomaly_points = time_series[is_anomaly == 1]
    fig = add_points(
        fig, anomaly_points.index, anomaly_points["value_0"], "Anomalies", "red"
    )
    fig = add_confidence_interval(fig, time_series)
    return fig

In [7]:
import plotly.express as px
from matplotlib.colors import to_rgb, to_hex

# Глобальная нормализация расстояний
distances = results['distance'].values
vmin, vmax = distances.min(), distances.max()

# Функция преобразования distance в цвет
def get_color(distance):
    # Нормируем от 0 до 1
    if vmax != vmin:
        t = (distance - vmin) / (vmax - vmin)
    else:
        t = 0.5
    cmap = plt.cm.get_cmap('RdYlGn_r')
    rgba = cmap(t)
    return f'rgba({int(rgba[0]*255)},{int(rgba[1]*255)},{int(rgba[2]*255)},{rgba[3]})'

grouped = results.groupby('series_id')

for series_id, group in grouped:
    full_series = searcher.prepared_series_[series_id]
    
    fig = go.Figure()
    add_line(fig, full_series.index, full_series.values, f"Весь ряд {series_id}", '#4d4d4d')
    
    # Для каждого сегмента добавляем заливку и линию
    for idx, row in group.iterrows():
        segment = searcher.get_segment(row['series_id'], int(row['start_idx']), int(row['end_idx']))
        color = get_color(row['distance'])
        # Заливка
        fig.add_vrect(
            x0=row['start_time'], x1=row['end_time'],
            fillcolor=color, opacity=0.25,
            layer="below", line_width=0
        )
        # Линия сегмента
        add_line(fig, segment.index, segment.values,
                 f"dist={row['distance']:.4f} (idx {row['start_idx']}:{row['end_idx']})",
                 color)
    
    update_layout(fig)
    fig.update_layout(
        title=f"Датасет: {series_id}",
        showlegend=True,
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02)
    )
    fig.show()

/tmp/ipykernel_1124/788950158.py:15: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('RdYlGn_r')


/tmp/ipykernel_1124/788950158.py:15: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('RdYlGn_r')


/tmp/ipykernel_1124/788950158.py:15: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('RdYlGn_r')


/tmp/ipykernel_1124/788950158.py:15: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('RdYlGn_r')
